In [1]:
# ============================================
# AI MEDICAL DOCTOR ASSISTANT
# Gemini API + Gradio UI
# Google Colab Ready
# ============================================

# INSTALL REQUIRED LIBRARIES
!pip install -q gradio google-genai pillow

# ============================================
# IMPORTS
# ============================================

import gradio as gr
from google import genai
from PIL import Image
import tempfile
import os

# ============================================
# GEMINI API CONFIGURATION
# ============================================

# ENTER YOUR GEMINI API KEY
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

# ============================================
# PROFESSIONAL SYSTEM PROMPT
# ============================================

SYSTEM_PROMPT = """
You are MedAssist AI, a highly professional AI medical assistant designed ONLY to assist licensed medical doctors and healthcare professionals.

Your responsibilities include:

1. Analyze patient symptoms
2. Analyze uploaded medical images
3. Generate possible differential diagnoses
4. Assess severity level
5. Recommend diagnostic tests
6. Suggest treatment considerations
7. Generate structured clinical notes
8. Identify emergency warning signs
9. Summarize findings professionally
10. Assist doctors in decision support

STRICT RULES:
- Never provide guaranteed diagnosis
- Never replace professional medical judgment
- Clearly mention uncertainty where appropriate
- Always include medical disclaimer
- Be professional and clinically structured
- Use medical terminology appropriately
- Mention urgent red flags if detected
- Keep outputs clear and organized

OUTPUT FORMAT:

# Patient Summary

# Possible Diagnoses

# Severity Assessment

# Recommended Diagnostic Tests

# Suggested Treatment Considerations

# Emergency Warning Signs

# Clinical Notes

# Medical Disclaimer

IMPORTANT:
This system assists doctors only and should not replace real clinical evaluation.
"""

# ============================================
# AI ANALYSIS FUNCTION
# ============================================

def analyze_medical_case(
    patient_name,
    patient_age,
    patient_gender,
    symptoms,
    medical_history,
    medications,
    uploaded_image,
    analysis_type
):

    try:

        prompt = f"""
        {SYSTEM_PROMPT}

        ANALYSIS TYPE:
        {analysis_type}

        PATIENT INFORMATION:

        Patient Name:
        {patient_name}

        Age:
        {patient_age}

        Gender:
        {patient_gender}

        Symptoms:
        {symptoms}

        Medical History:
        {medical_history}

        Current Medications:
        {medications}

        Please provide a detailed professional medical analysis.
        """

        contents = [prompt]

        # IMAGE PROCESSING
        if uploaded_image is not None:

            temp_file = tempfile.NamedTemporaryFile(
                suffix=".png",
                delete=False
            )

            image = Image.open(uploaded_image)
            image.save(temp_file.name)

            uploaded_file = client.files.upload(
                file=temp_file.name
            )

            contents.append(uploaded_file)

        # GEMINI RESPONSE
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=contents
        )

        return response.text

    except Exception as e:
        return f"Error: {str(e)}"

# ============================================
# PROFESSIONAL GRADIO UI
# ============================================

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="AI Medical Doctor Assistant"
) as demo:

    gr.Markdown("""
    # 🏥 AI Medical Doctor Assistant

    ### Gemini Multimodal AI for Healthcare Professionals

    Features:
    - Symptom Analysis
    - Medical Image Analysis
    - Clinical Note Generation
    - Differential Diagnosis Support
    - Emergency Detection
    - Treatment Suggestions
    """)

    with gr.Tabs():

        # ====================================
        # MAIN ANALYSIS TAB
        # ====================================

        with gr.Tab("🩺 Patient Analysis"):

            with gr.Row():

                with gr.Column(scale=1):

                    patient_name = gr.Textbox(
                        label="Patient Name",
                        placeholder="Enter patient name"
                    )

                    patient_age = gr.Number(
                        label="Patient Age",
                        value=30
                    )

                    patient_gender = gr.Dropdown(
                        choices=[
                            "Male",
                            "Female",
                            "Other"
                        ],
                        label="Gender"
                    )

                    analysis_type = gr.Dropdown(
                        choices=[
                            "General Diagnosis",
                            "Radiology Review",
                            "Skin Condition Analysis",
                            "Emergency Assessment",
                            "Prescription Guidance",
                            "Clinical Notes Generation"
                        ],
                        value="General Diagnosis",
                        label="Analysis Type"
                    )

                with gr.Column(scale=2):

                    symptoms = gr.Textbox(
                        lines=6,
                        label="Symptoms",
                        placeholder="Describe patient symptoms in detail..."
                    )

                    medical_history = gr.Textbox(
                        lines=4,
                        label="Medical History",
                        placeholder="Diabetes, hypertension, allergies, surgeries..."
                    )

                    medications = gr.Textbox(
                        lines=3,
                        label="Current Medications",
                        placeholder="List current medications..."
                    )

            gr.Markdown("## 📷 Medical Image Upload")

            uploaded_image = gr.Image(
                type="filepath",
                label="Upload Medical Image"
            )

            analyze_btn = gr.Button(
                "🔍 Analyze Patient Case",
                variant="primary"
            )

            output = gr.Markdown(
                label="AI Medical Analysis"
            )

            analyze_btn.click(
                fn=analyze_medical_case,
                inputs=[
                    patient_name,
                    patient_age,
                    patient_gender,
                    symptoms,
                    medical_history,
                    medications,
                    uploaded_image,
                    analysis_type
                ],
                outputs=output
            )

        # ====================================
        # RADIOLOGY TAB
        # ====================================

        with gr.Tab("🩻 Radiology Assistant"):

            gr.Markdown("""
            ## Radiology AI Assistant

            Upload:
            - X-rays
            - CT scans
            - MRI scans
            - Ultrasound images
            """)

            radio_image = gr.Image(
                type="filepath",
                label="Upload Radiology Image"
            )

            radio_notes = gr.Textbox(
                lines=5,
                label="Clinical Notes"
            )

            radio_output = gr.Markdown()

            def radiology_analysis(img, notes):

                return analyze_medical_case(
                    patient_name="Radiology Patient",
                    patient_age=0,
                    patient_gender="Unknown",
                    symptoms=notes,
                    medical_history="",
                    medications="",
                    uploaded_image=img,
                    analysis_type="Radiology Review"
                )

            radio_btn = gr.Button(
                "Analyze Radiology Scan"
            )

            radio_btn.click(
                radiology_analysis,
                inputs=[
                    radio_image,
                    radio_notes
                ],
                outputs=radio_output
            )

        # ====================================
        # CLINICAL NOTES TAB
        # ====================================

        with gr.Tab("📄 Clinical Notes Generator"):

            notes_input = gr.Textbox(
                lines=10,
                label="Doctor Notes / Consultation Notes"
            )

            notes_output = gr.Markdown()

            def generate_notes(text):

                prompt = f"""
                Generate professional structured clinical notes from:

                {text}

                Include:
                - Chief complaint
                - Assessment
                - Findings
                - Recommendations
                """

                response = client.models.generate_content(
                    model="gemini-2.5-flash",
                    contents=prompt
                )

                return response.text

            notes_btn = gr.Button(
                "Generate Clinical Notes"
            )

            notes_btn.click(
                generate_notes,
                inputs=notes_input,
                outputs=notes_output
            )

        # ====================================
        # EMERGENCY TRIAGE TAB
        # ====================================

        with gr.Tab("🚨 Emergency Triage"):

            emergency_symptoms = gr.Textbox(
                lines=8,
                label="Emergency Symptoms"
            )

            emergency_output = gr.Markdown()

            def emergency_analysis(text):

                prompt = f"""
                You are an emergency triage AI assistant.

                Analyze:
                {text}

                Determine:
                - Severity
                - Immediate risk
                - Emergency level
                - Recommended immediate action

                Use professional emergency medicine format.
                """

                response = client.models.generate_content(
                    model="gemini-2.5-flash",
                    contents=prompt
                )

                return response.text

            emergency_btn = gr.Button(
                "Assess Emergency"
            )

            emergency_btn.click(
                emergency_analysis,
                inputs=emergency_symptoms,
                outputs=emergency_output
            )

        # ====================================
        # ABOUT TAB
        # ====================================

        with gr.Tab("ℹ️ About"):

            gr.Markdown("""
            ## AI Medical Doctor Assistant

            ### Powered By
            - Gemini Multimodal AI
            - Gradio
            - Google Colab

            ### Features
            ✅ Symptom Analysis
            ✅ Medical Imaging
            ✅ Clinical Note Generation
            ✅ Emergency Detection
            ✅ Diagnostic Assistance
            ✅ Professional Medical Workflow

            ### Important Disclaimer

            This system is designed ONLY to assist licensed healthcare professionals.

            It must NOT replace:
            - Real medical diagnosis
            - Clinical judgment
            - Emergency medical care

            Always verify AI-generated outputs clinically.
            """)

# ============================================
# LAUNCH APPLICATION
# ============================================

demo.launch(
    debug=True,
    share=True
)

/tmp/ipykernel_14986/1873072221.py:161: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://121a06d2f2067c4d04.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://121a06d2f2067c4d04.gradio.live
